In [ ]:
# 🧪 BioDockify v3: ULTIMATE ALZHEIMER'S MODEL
# - 3 Targets (AChE, BACE1, GSK3b)
# - UNLIMITED Data Download
# - 5-Fold Rigorous Cross-Validation
# 1. SETUP
try:
    import torch, transformers, rdkit
except:
    !pip install transformers accelerate datasets torch rdkit scikit-learn chembl_webresource_client
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModel, AutoTokenizer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from chembl_webresource_client.new_client import new_client
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
import os
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f">>> 🚀 v3 TRAINING ENGINE STARTED ON {DEVICE}")
# 2. MAX DATA DOWNLOAD (3 TARGETS)
if not os.path.exists("chembl_alzheimers.csv"):
    print(">>> ⬇️ Downloading FULL Alzheimer's Data (AChE, BACE1, GSK3b)...")
    activity = new_client.activity
    targets = [
        {'id': 'CHEMBL220', 'name': 'AChE'},
        {'id': 'CHEMBL340', 'name': 'BACE1'},
        {'id': 'CHEMBL262', 'name': 'GSK3b'}
    ]
    all_data = []

    for t in targets:
        print(f"    Fetching {t['name']} ({t['id']})...")
        # NO LIMITS applied here
        res = activity.filter(target_chembl_id=t['id'], pchembl_value__isnull=False).only('canonical_smiles', 'pchembl_value')

        count = 0
        for act in tqdm(res, desc=t['name']):
            if act.get('canonical_smiles'):
                all_data.append({
                    'smiles': act['canonical_smiles'],
                    'pIC50': float(act['pchembl_value']),
                    'target': t['name']
                })
                count += 1
        print(f"    ✅ Got {count} compounds for {t['name']}")

    df = pd.DataFrame(all_data)
    df.to_csv("chembl_alzheimers.csv", index=False)
    print(f"✅ FINAL DATASET: {len(df)} Compounds Saved.")
# 3. PREP
df = pd.read_csv("chembl_alzheimers.csv")
y = np.array((df['pIC50'] >= 7.0).astype(int).tolist())
X = np.array(df['smiles'].tolist())
n_pos = sum(y)
ratio = (len(y) - n_pos) / n_pos if n_pos > 0 else 1.0
print(f"📊 Training on {len(y)} compounds (Actives={n_pos} | Inactives={len(y)-n_pos} | W={ratio:.1f})")
# 4. MODEL
MODEL_NAME = "ibm/MoLFormer-XL-both-10pct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
class SpecialistModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_NAME, trust_remote_code=True)
        self.classifier = nn.Sequential(
            nn.Linear(768, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Linear(128, 1)
        )
    def forward(self, ids, mask):
        emb = self.encoder(ids, mask).pooler_output
        return self.classifier(emb)
# 5. RIGOROUS TRAINING
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
best_f1 = 0.0
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n🔄 FOLD {fold+1}/5 STARTED...")
    model = SpecialistModel().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([ratio]).to(DEVICE))

    # Dynamic Tokenization to save RAM with large datasets
    X_tr, X_val = X[train_idx], X[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]

    # Train Loop (Simplified for Notebook)
    model.train()
    enc = tokenizer(X_tr.tolist(), padding='max_length', truncation=True, max_length=128, return_tensors='pt')
    ds = TensorDataset(enc['input_ids'], enc['attention_mask'], torch.tensor(y_tr, dtype=torch.float))
    dl = DataLoader(ds, batch_size=32, shuffle=True)

    for epoch in range(3):
        total_loss = 0
        for ids, mask, lbl in dl:
            optimizer.zero_grad()
            logits = model(ids.to(DEVICE), mask.to(DEVICE))
            loss = criterion(logits, lbl.unsqueeze(1).to(DEVICE))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"    Epoch {epoch+1}: Loss {total_loss/len(dl):.4f}")

    # Validate
    model.eval()
    enc_v = tokenizer(X_val.tolist(), padding='max_length', truncation=True, max_length=128, return_tensors='pt')
    with torch.no_grad():
        logits = model(enc_v['input_ids'].to(DEVICE), enc_v['attention_mask'].to(DEVICE))
        probs = torch.sigmoid(logits).cpu().numpy()
        preds = [1 if p > 0.5 else 0 for p in probs]
        f1 = f1_score(y_val, preds)

    print(f"    ✅ FOLD {fold+1} F1-SCORE: {f1:.4f}")
    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), "biodockify_v3_master.pth")
print(f"\n🏆 COMPLETED. Best Model Saved ('biodockify_v3_master.pth') with F1: {best_f1:.4f}")

>>> 🚀 v3 TRAINING ENGINE STARTED ON cuda
>>> ⬇️ Downloading FULL Alzheimer's Data (AChE, BACE1, GSK3b)...
    Fetching AChE (CHEMBL220)...


AChE:   0%|          | 0/7941 [00:00<?, ?it/s]

    ✅ Got 7940 compounds for AChE
    Fetching BACE1 (CHEMBL340)...


BACE1:   0%|          | 0/29257 [00:00<?, ?it/s]

HttpApplicationError: Error for url https://www.ebi.ac.uk/chembl/api/data/activity.json, server response: <!doctype html>
<html lang="en" class="vf-no-js">
  <head>
    <script>
// Detect if JS is on and swap vf-no-js for vf-js on the html element
(function(H){H.className=H.className.replace(/\bvf-no-js\b/,'vf-js')})(document.documentElement);
</script>

    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <!-- <link rel="stylesheet" media="all" href="/css/styles.css?" /> -->
    <title>Error: 500 | EMBLâs European Bionformatics Institute</title>



    <link rel="icon" type="image/x-icon"
  href="https://ebi.emblstatic.net/web_guidelines/EBI-Framework/v1.4/images/logos/EMBL-EBI/favicons/favicon.ico" />
<link rel="icon" type="image/png"
  href="https://ebi.emblstatic.net/web_guidelines/EBI-Framework/v1.4/images/logos/EMBL-EBI/favicons/favicon-32x32.png" />
<link rel="icon" type="image/png" sizes="192Ã192"
  href="https://ebi.embls